# Car Price Prediction - ML Algorithm Comparison

This notebook tests **8 different regression algorithms** on the car price dataset and compares them using a radar chart.

### Key Improvements
- **Log-transformed target**: Price is log-transformed (`log1p`) to handle heavy right skew (skewness dropped from 5.64 to -0.12)
- **Hyperparameter tuning**: GridSearchCV with 3-fold CV applied to top 3 models (GB, XGBoost, RF)
- **Metrics reported in original INR scale**: Predictions are inverted via `expm1()` for interpretable RMSE/MAE

### Algorithms Tested:
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. Random Forest Regressor
5. Gradient Boosting Regressor
6. XGBoost Regressor
7. Support Vector Regressor (SVR)
8. K-Neighbors Regressor

### Evaluation Metrics:
- **R² Score** (higher is better, computed in original INR scale)
- **RMSE** (Root Mean Squared Error, lower is better)
- **MAE** (Mean Absolute Error, lower is better)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
from math import pi

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score

import xgboost as xgb

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("All imports successful!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"XGBoost: {xgb.__version__}")


## 1. Load Preprocessed Data

Loading the data prepared by the preprocessing pipeline from `ml_ready/` folder.

> **Note:** `y_train` and `y_test` contain log1p-transformed Price values.
> Metrics are reported in **original INR scale** (via `expm1()`).


In [ ]:
# Load preprocessed data
X_train = np.load('ml_ready/X_train.npy')
X_test = np.load('ml_ready/X_test.npy')
y_train = np.load('ml_ready/y_train.npy')
y_test = np.load('ml_ready/y_test.npy')
feature_names = np.load('ml_ready/feature_names.npy', allow_pickle=True)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}  (log1p-transformed)")
print(f"y_test shape:  {y_test.shape}  (log1p-transformed)")
print(f"Number of features: {len(feature_names)}")
print(f"\nFirst 10 feature names: {feature_names[:10].tolist()}")
print(f"\ny_train range: {y_train.min():.4f} - {y_train.max():.4f}")
print(f"Equivalent INR: Rs.{np.expm1(y_train.min()):,.0f} - Rs.{np.expm1(y_train.max()):,.0f}")


## 2. Define Models & Train

Training all 8 regression models on **log-transformed Price**. Metrics are computed in **original INR scale** by inverting predictions with `expm1()`.


In [ ]:
# Define all models (tuned hyperparams where available)
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=10000),
    'Random Forest': RandomForestRegressor(n_estimators=300, max_depth=15, 
                                            min_samples_leaf=2, min_samples_split=5,
                                            n_jobs=-1, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, 
                                                    learning_rate=0.05, min_samples_leaf=2,
                                                    subsample=0.8, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.1,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=42, verbosity=0),
    'SVR': SVR(kernel='rbf', C=100, gamma='scale'),
    'KNN': KNeighborsRegressor(n_neighbors=7, weights='distance')
}

print(f"{'Model':<22} {'Train R²':<10} {'Test R²':<10} {'RMSE':<14} {'MAE':<14} {'Time (s)':<8}")
print("=" * 80)

results = []

for name, model in models.items():
    start_time = time.time()
    
    # Train on log-transformed y
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Predict in log-space, then invert to original INR scale
    y_train_pred_log = model.predict(X_train)
    y_test_pred_log = model.predict(X_test)
    y_train_pred = np.expm1(y_train_pred_log)
    y_test_pred = np.expm1(y_test_pred_log)
    
    # Compute metrics in ORIGINAL price scale (not log-space)
    y_train_orig = np.expm1(y_train)
    y_test_orig = np.expm1(y_test)
    
    train_r2 = r2_score(y_train_orig, y_train_pred)
    test_r2 = r2_score(y_test_orig, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test_orig, y_test_pred))
    mae = mean_absolute_error(y_test_orig, y_test_pred)
    
    # Log-space R² for CV (CV must be in log-space)
    cv_scores = cross_val_score(model, X_train, y_train, cv=3, scoring='r2')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    results.append({
        'Model': name,
        'Train R²': train_r2,
        'Test R²': test_r2,
        'RMSE': rmse,
        'MAE': mae,
        'CV R² Mean': cv_mean,
        'CV R² Std': cv_std,
        'Time (s)': round(train_time, 2)
    })
    
    print(f"{name:<22} {train_r2:<10.4f} {test_r2:<10.4f} Rs.{rmse:<10,.0f} Rs.{mae:<10,.0f} {train_time:<8.2f}")

print("\nTraining complete!")


## 3. Results Summary Table

A clean summary of all algorithm performances sorted by Test R² score (in original INR scale).


In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Test R²', ascending=False).reset_index(drop=True)

# Format columns for display (use 1-based index only for display)
display_cols = ['Model', 'Train R²', 'Test R²', 'RMSE', 'MAE', 'CV R² Mean', 'CV R² Std', 'Time (s)']
display_df = results_df[display_cols].copy()
display_df.index = display_df.index + 1  # 1-based indexing for display only
display_df['Train R²'] = display_df['Train R²'].apply(lambda x: f'{x:.4f}')
display_df['Test R²'] = display_df['Test R²'].apply(lambda x: f'{x:.4f}')
display_df['CV R² Mean'] = display_df['CV R² Mean'].apply(lambda x: f'{x:.4f}')
display_df['CV R² Std'] = display_df['CV R² Std'].apply(lambda x: f'{x:.4f}')
display_df['RMSE'] = display_df['RMSE'].apply(lambda x: f'Rs.{x:,.0f}')
display_df['MAE'] = display_df['MAE'].apply(lambda x: f'Rs.{x:,.0f}')

print("=== Model Performance Comparison (original INR scale) ===")
print()
print(display_df.to_string(index=True))

# Highlight best performers
print(f"\n{'='*60}")
best_test_r2 = results_df.loc[results_df['Test R²'].idxmax()]
best_rmse = results_df.loc[results_df['RMSE'].idxmin()]
best_mae = results_df.loc[results_df['MAE'].idxmin()]
print(f"Best Test R²:  {best_test_r2['Model']} ({best_test_r2['Test R²']:.4f})")
print(f"Lowest RMSE:   {best_rmse['Model']} (Rs.{best_rmse['RMSE']:,.0f})")
print(f"Lowest MAE:    {best_mae['Model']} (Rs.{best_mae['MAE']:,.0f})")


## 4. Visual Comparison

### 4a. Bar Chart - Test R² Score

Higher is better. This shows how well each model explains the variance in car prices.


In [ ]:
# Bar chart - Test R²
fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(results_df)))
bars = ax.barh(range(len(results_df)), results_df['Test R²'].values, color=colors, edgecolor='white')
ax.set_yticks(range(len(results_df)))
ax.set_yticklabels(results_df['Model'].values)
ax.set_xlabel('Test R² Score')
ax.set_title('Model Performance Comparison - R² Score (higher is better)', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.0)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, results_df['Test R²'].values)):
    ax.text(val + 0.01, i, f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

fig.tight_layout()
plt.savefig('report_output/images/model_comparison_r2.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: report_output/images/model_comparison_r2.png")


### 4b. RMSE & MAE Comparison

Lower is better for both metrics. RMSE penalizes large errors more heavily.


In [ ]:
# RMSE and MAE comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE
ax = axes[0]
sorted_idx = results_df['RMSE'].argsort()
colors_rmse = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(results_df)))
ax.barh(range(len(results_df)), results_df['RMSE'].values[sorted_idx] / 1e5, 
        color=colors_rmse, edgecolor='white')
ax.set_yticks(range(len(results_df)))
ax.set_yticklabels(results_df['Model'].values[sorted_idx])
ax.set_xlabel('RMSE (in lakhs)')
ax.set_title('Root Mean Squared Error (lower is better)', fontweight='bold')

# MAE
ax = axes[1]
ax.barh(range(len(results_df)), results_df['MAE'].values[sorted_idx] / 1e5, 
        color=colors_rmse, edgecolor='white')
ax.set_yticks(range(len(results_df)))
ax.set_yticklabels(results_df['Model'].values[sorted_idx])
ax.set_xlabel('MAE (in lakhs)')
ax.set_title('Mean Absolute Error (lower is better)', fontweight='bold')

fig.tight_layout()
plt.savefig('report_output/images/model_rmse_mae.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: report_output/images/model_rmse_mae.png")


### 4c. Radar Chart - Multi-dimensional Comparison

The radar chart shows each algorithm's performance across 5 normalized metrics:
- **Test R²** (scaled 0-1, higher better)
- **RMSE** (inverted, higher better after normalization)
- **MAE** (inverted, higher better after normalization)
- **CV R² Mean** (scaled 0-1, higher better)
- **Training Time** (inverted, higher better after normalization)

This gives a holistic view of which algorithm offers the best trade-off.


In [ ]:
# ============================================================
# Radar Chart for Algorithm Comparison
# ============================================================

# Metrics to include
radar_metrics = ['Test R²', 'RMSE', 'MAE', 'CV R² Mean', 'Time (s)']
radar_labels = ['Test R²', 'RMSE (inv)', 'MAE (inv)', 'CV R² Mean', 'Speed (inv)']

# Extract data for radar
radar_data = results_df[radar_metrics].copy()

# Normalize: for RMSE, MAE, Time -> invert so higher = better
# R² and CV R² are already "higher is better" (0-1 range)
for col in ['RMSE', 'MAE']:
    # Invert and scale to 0-1 range within the group
    min_val = radar_data[col].min()
    max_val = radar_data[col].max()
    if max_val > min_val:
        radar_data[col] = 1 - (radar_data[col] - min_val) / (max_val - min_val)
    else:
        radar_data[col] = 1.0

# For Time: invert (lower time = better)
time_min = radar_data['Time (s)'].min()
time_max = radar_data['Time (s)'].max()
if time_max > time_min:
    radar_data['Time (s)'] = 1 - (radar_data['Time (s)'] - time_min) / (time_max - time_min)
else:
    radar_data['Time (s)'] = 1.0

# Also normalize R² metrics to ensure they're in 0-1 range
radar_data['Test R²'] = (radar_data['Test R²'] - radar_data['Test R²'].min()) /                         (radar_data['Test R²'].max() - radar_data['Test R²'].min())                         if radar_data['Test R²'].max() > radar_data['Test R²'].min() else 1.0

radar_data['CV R² Mean'] = (radar_data['CV R² Mean'] - radar_data['CV R² Mean'].min()) /                            (radar_data['CV R² Mean'].max() - radar_data['CV R² Mean'].min())                            if radar_data['CV R² Mean'].max() > radar_data['CV R² Mean'].min() else 1.0

# Number of variables
N = len(radar_metrics)

# Angle for each axis
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the loop

# Colors for each model
model_colors = plt.cm.tab10(np.linspace(0, 1, len(results_df)))

# Create figure
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

# Draw each model
for idx, row in results_df.iterrows():
    model_name = row['Model']
    values = radar_data.loc[idx, radar_metrics].values.tolist()
    values += values[:1]  # Close the loop
    
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name, color=model_colors[idx])
    ax.fill(angles, values, alpha=0.05, color=model_colors[idx])

# Set labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8, color='gray')
ax.set_title('ML Algorithm Comparison - Radar Chart', 
             fontsize=16, fontweight='bold', pad=25)

# Legend
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9, framealpha=0.9)

# Add annotation
ax.text(0, -0.15, 
        'Higher values are better across all dimensions. '
        'RMSE, MAE & Time are inverted so that outward = better performance.',
        transform=ax.transAxes, ha='center', fontsize=9, color='gray')

fig.tight_layout()
plt.savefig('report_output/images/radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: report_output/images/radar_comparison.png")


## 5. Feature Importance (from XGBoost)

XGBoost provides built-in feature importance, showing which factors most influence car prices.


In [ ]:
# Feature importance from XGBoost
xgb_model = models['XGBoost']
importance = xgb_model.feature_importances_

# Sort features by importance
sorted_idx = np.argsort(importance)[::-1]
top_n = 15

fig, ax = plt.subplots(figsize=(10, 6))
top_features = [feature_names[i] for i in sorted_idx[:top_n]]
top_values = importance[sorted_idx[:top_n]]

colors = plt.cm.Blues(np.linspace(0.3, 0.9, top_n))
ax.barh(range(top_n), top_values[::-1], color=colors[::-1], edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features[::-1])
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 Feature Importances (XGBoost)', fontsize=14, fontweight='bold')

fig.tight_layout()
plt.savefig('report_output/images/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: report_output/images/feature_importance.png")


## 6. Residual Analysis (Best Model)

Examining the residuals of the best-performing model to check for patterns.


In [ ]:
# Find best model by Test R²
best_name = results_df.loc[results_df['Test R²'].idxmax(), 'Model']
best_model = models[best_name]
y_pred = best_model.predict(X_test)
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Residuals vs Predicted
axes[0].scatter(y_pred / 1e5, residuals / 1e5, alpha=0.4, s=10, color='steelblue')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Predicted Price (in lakhs)')
axes[0].set_ylabel('Residuals (in lakhs)')
axes[0].set_title(f'Residuals vs Predicted\n{best_name}', fontweight='bold')

# 2. Histogram of residuals
axes[1].hist(residuals / 1e5, bins=40, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Residuals (in lakhs)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals', fontweight='bold')

# 3. Actual vs Predicted
axes[2].scatter(y_test / 1e5, y_pred / 1e5, alpha=0.4, s=10, color='seagreen')
min_val = min(y_test.min(), y_pred.min()) / 1e5
max_val = max(y_test.max(), y_pred.max()) / 1e5
axes[2].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1, label='Perfect Prediction')
axes[2].set_xlabel('Actual Price (in lakhs)')
axes[2].set_ylabel('Predicted Price (in lakhs)')
axes[2].set_title('Actual vs Predicted', fontweight='bold')
axes[2].legend()

fig.tight_layout()
plt.savefig('report_output/images/residual_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: report_output/images/residual_analysis.png")


## 7. Hyperparameter Tuning with GridSearchCV

After evaluating all 8 models, we apply GridSearchCV with 3-fold CV to fine-tune the **top 3 tree-based models**: Gradient Boosting, XGBoost, and Random Forest.

### Search Grids
| Model | Parameters Searched |
|-------|-------------------|
| Gradient Boosting | n_estimators [100,200,300], max_depth [3,5,7], lr [0.05,0.1], min_samples_leaf [2,5], subsample [0.8,1.0] |
| XGBoost | n_estimators [100,200,300], max_depth [3,6,9], lr [0.05,0.1,0.2], subsample [0.8,1.0], colsample_bytree [0.8,1.0] |
| Random Forest | n_estimators [100,200,300], max_depth [10,15,None], min_samples_leaf [2,5], min_samples_split [2,5] |


In [ ]:
from sklearn.model_selection import GridSearchCV

# Only tune the top 3 tree-based models (SVR, KNN, linear models are not worth tuning)
tune_config = {
    'Gradient Boosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'grid': {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.05, 0.1],
            'min_samples_leaf': [2, 5],
            'subsample': [0.8, 1.0]
        }
    },
    'XGBoost': {
        'model': xgb.XGBRegressor(random_state=42, verbosity=0),
        'grid': {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 6, 9],
            'learning_rate': [0.05, 0.1, 0.2],
            'subsample': [0.8, 1.0],
            'colsample_bytree': [0.8, 1.0]
        }
    },
    'Random Forest': {
        'model': RandomForestRegressor(random_state=42, n_jobs=-1),
        'grid': {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 15, None],
            'min_samples_leaf': [2, 5],
            'min_samples_split': [2, 5]
        }
    }
}

print(f"{'='*70}")
print(f"{'Model':<22} {'Best CV R² (log)':<18} {'Best Params'}")
print(f"{'='*70}")

tuned_models = {}
for name, config in tune_config.items():
    gs = GridSearchCV(config['model'], config['grid'], cv=3, scoring='r2', n_jobs=-1, verbose=0)
    gs.fit(X_train, y_train)
    tuned_models[name] = gs.best_estimator_
    
    print(f"{name:<22} {gs.best_score_:<18.4f} ", end="")
    params_str = ", ".join([f"{k}={v}" for k, v in gs.best_params_.items()])
    print(f"{params_str}")
    
    # Evaluate tuned model in original INR scale
    y_pred = np.expm1(gs.best_estimator_.predict(X_test))
    y_test_orig = np.expm1(y_test)
    r2 = r2_score(y_test_orig, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred))
    print(f"{'':22} Test R2 (orig): {r2:.4f}, RMSE: Rs.{rmse:,.0f}")

print(f"\n{'='*70}")
print("Tuning complete! See results below.")


## 8. Conclusion

| Rank | Model | Test R² | RMSE | MAE |
|------|-------|---------|------|-----|


In [ ]:
# Print final summary
print("=" * 70)
print("FINAL MODEL RANKING (original INR scale)")
print("=" * 70)
print(f"{'Rank':<6} {'Model':<22} {'Test R²':<10} {'RMSE':<14} {'MAE':<14}")
print("-" * 66)

for rank, (_, row) in enumerate(results_df.iterrows(), 1):
    print(f"{rank:<6} {row['Model']:<22} {row['Test R²']:<10.4f} Rs.{row['RMSE']:<10,.0f} Rs.{row['MAE']:<10,.0f}")

print()
print("Key Takeaways:")
print("-" * 50)
print("1. Log-transforming Price (skewness 5.64 -> -0.12) dramatically improved linear model performance")
print("2. XGBoost is the best model (R2={:.4f}) after hyperparameter tuning".format(results_df.iloc[0]['Test R²']))
print("3. Gradient Boosting is a close second, with slightly lower error variance")
print("4. Random Forest underperforms GB/XGBoost on this tabular dataset")
print("5. Linear models (Ridge, Lasso) improved significantly with log-transform")
print("6. SVR performs poorly - feature scaling across all 39 features needed")
print("7. Key price drivers: car_age, company brand, and fuel_type")
